# Principal Stress Branching Pattern — v3 (Improved)

## What this notebook does
It generates a branching pattern on a triangular mesh that is guided by the **principal stress field**.
Think of it like the veins in a leaf — the thicker veins carry more load, and they split into thinner ones
as they spread outward.

## Pipeline overview
```
1. Generate triangular mesh (nodes + triangles)
2. Compute principal stress field analytically (σ₁, σ₂, directions)
3. Build a spanning tree using Prim's algorithm with stress-guided edge weights
4. Map σ₁ magnitude → branch linewidth
5. Visualise
```

## Key improvements in v3
| Issue in v2 | Fix in v3 |
|---|---|
| ~20% of branches went backward (stubs) | Added **backward-motion penalty** δ to edge weight |
| Branch thickness was only depth-based | Branch thickness now maps to **σ₁ magnitude** |
| Only σ₁ direction used | **σ₂ direction blended in** (20%) for richer branching geometry |
| Tiny micro-edges created stubs | **Minimum edge length** filter added |

## Edge weight formula (the core idea)
```
w(u→v) = α · (1 − |align with σ₁|)   ← penalise misalignment with stress direction
        + β · edge_length              ← penalise long edges
        − γ · stress_magnitude         ← reward edges in high-stress zones
        + δ · backward_penalty         ← penalise edges going against current direction
```
Lower weight = preferred edge. The algorithm always picks the cheapest available edge.

In [ ]:
# ─────────────────────────────────────────────────────────
# IMPORTS — standard scientific Python libraries
# numpy    : arrays and maths
# matplotlib: plotting
# scipy    : Delaunay triangulation
# heapq    : priority queue for Prim's algorithm
# ─────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from scipy.spatial import Delaunay
from collections import defaultdict
import heapq
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 130})
print('All libraries loaded ✓')

---
## Step 1 — Triangular Mesh Generator

In [ ]:
def generate_mesh(Lx, Ly, n_approx=800, jitter=0.28, seed=42):
    """
    Create a triangular mesh that fills the rectangle [0,Lx] × [0,Ly].

    HOW IT WORKS:
      1. Lay down a regular grid of points.
      2. Add small random offsets (jitter) to interior points
         so triangles are irregular — more like a real mesh.
      3. Connect points into triangles using Delaunay triangulation.
      4. Build an adjacency dictionary so each node knows its neighbours.

    Parameters
    ----------
    Lx, Ly    : width and height of the rectangle
    n_approx  : approximate number of nodes (controls mesh density)
    jitter    : random displacement as a fraction of grid spacing
                0.0 = perfectly regular grid
                0.4 = quite organic-looking
    seed      : random seed — same seed always gives same mesh

    Returns
    -------
    nodes     : (N, 2) array — x,y coordinates of every node
    triangles : (T, 3) array — three node indices forming each triangle
    valence   : (N,)   array — how many edges meet at each node
    adj       : dict   — adj[i] = set of node indices connected to node i
    """

    # Work out grid dimensions to hit the target node count
    aspect = Lx / Ly
    ny = int(np.sqrt(n_approx / aspect)) + 1
    nx = int(aspect * ny) + 1
    dx = Lx / (nx - 1)   # grid spacing in x
    dy = Ly / (ny - 1)   # grid spacing in y

    # Build the regular grid
    xs = np.linspace(0, Lx, nx)
    ys = np.linspace(0, Ly, ny)
    gx, gy = np.meshgrid(xs, ys)
    pts = np.column_stack([gx.ravel(), gy.ravel()])

    # Flag boundary nodes — these stay fixed (no jitter)
    on_boundary = (
        (pts[:, 0] < 1e-10) | (pts[:, 0] > Lx - 1e-10) |
        (pts[:, 1] < 1e-10) | (pts[:, 1] > Ly - 1e-10)
    )

    # Add random jitter to interior nodes
    rng   = np.random.default_rng(seed)
    noise = rng.uniform(-jitter, jitter, pts.shape) * np.array([dx, dy])
    noise[on_boundary] = 0.0                      # freeze boundary
    pts = np.clip(pts + noise, [0, 0], [Lx, Ly])  # stay inside rectangle

    # Delaunay triangulation — connect dots into triangles
    tri_obj   = Delaunay(pts)
    triangles = tri_obj.simplices

    # Remove degenerate triangles (nearly zero area — numerical noise)
    def tri_area(t):
        a, b, c = pts[t[0]], pts[t[1]], pts[t[2]]
        return abs((b[0]-a[0])*(c[1]-a[1]) - (c[0]-a[0])*(b[1]-a[1])) / 2

    triangles = np.array([t for t in triangles if tri_area(t) > 1e-6 * dx * dy])

    # Build adjacency: for each node, which other nodes is it connected to?
    adj = defaultdict(set)
    for t in triangles:
        for i in range(3):
            a, b = t[i], t[(i + 1) % 3]
            adj[a].add(b)
            adj[b].add(a)

    # Valence = number of edges at each node
    valence = np.array([len(adj[i]) for i in range(len(pts))])

    return pts, triangles, valence, adj


# --- Quick test ---
test_nodes, test_tris, test_val, test_adj = generate_mesh(1.0, 1.0, 800)
print(f'Nodes: {len(test_nodes)}')
print(f'Triangles: {len(test_tris)}')
print(f'Valence range: {test_val.min()} – {test_val.max()}  (typical mesh: 5–7)')

# Visualise the mesh
fig, ax = plt.subplots(figsize=(6, 6), facecolor='#080810')
ax.set_facecolor('#080810')
ax.triplot(test_nodes[:,0], test_nodes[:,1], test_tris,
           color='#4444AA', linewidth=0.5, alpha=0.8)
sc = ax.scatter(test_nodes[:,0], test_nodes[:,1],
                c=test_val, cmap='plasma', s=10, zorder=2)
plt.colorbar(sc, ax=ax, label='Valence (edges per node)').ax.yaxis.set_tick_params(color='white')
ax.set_title('Triangular Mesh (coloured by valence)', color='white', fontweight='bold')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#3a3a5c')
plt.tight_layout(); plt.show()

---
## Step 2 — Analytical Principal Stress Fields

For each load case we use **Mohr's circle** to find:
- `sigma1` — maximum principal stress magnitude
- `sigma2` — minimum principal stress magnitude  
- `vec1`   — unit vector pointing in the `sigma1` direction
- `vec2`   — unit vector pointing in the `sigma2` direction (always ⊥ to vec1)

The branching tree will try to **follow `vec1`** — the direction of maximum principal stress.

In [ ]:
def stress_radial(nodes, cx=0.5, cy=0.5):
    """
    RADIAL stress field — like spokes radiating from a wheel hub.

    Physical meaning: stress in a flat plate with a central point load.
    Sigma1 (tension) points radially outward from the source.
    Sigma2 (compression) acts in the tangential direction.
    Stress magnitude decreases with distance from the centre.

    Parameters: cx, cy = centre of the source point
    """
    dx = nodes[:, 0] - cx
    dy = nodes[:, 1] - cy
    r  = np.sqrt(dx**2 + dy**2) + 1e-8   # distance from source (avoid zero)

    # Radial direction = from centre toward each node
    theta = np.arctan2(dy, dx)
    vec1  = np.column_stack([ np.cos(theta),  np.sin(theta)])  # radial
    vec2  = np.column_stack([-np.sin(theta),  np.cos(theta)])  # tangential (⊥)

    sigma1 = 1.0 / (r + 0.1)      # tension radially — high near centre
    sigma2 = -0.3 * sigma1         # mild compression tangentially

    return sigma1, sigma2, vec1, vec2


def stress_compression(nodes, Lx=1.0, Ly=2.0):
    """
    UNIAXIAL COMPRESSION stress field — like a column being squashed.

    Physical meaning: a column fixed at the base, loaded vertically at the top.
    Sigma_y = -1 everywhere (uniform vertical compression).
    A small sinusoidal shear is added to create spatial variation,
    giving the branching tree interesting paths to follow.

    Mohr's circle converts (sigma_x, sigma_y, tau) → principal stresses.
    """
    x, y = nodes[:, 0], nodes[:, 1]

    # Stress tensor components
    sig_x = np.zeros(len(nodes))       # no horizontal stress
    sig_y = -np.ones(len(nodes))       # uniform vertical compression
    tau   = 0.15 * np.sin(2*np.pi*x/Lx) * np.cos(np.pi*y/Ly)  # small shear perturbation

    # Mohr's circle → principal stresses
    avg    = (sig_x + sig_y) / 2
    R      = np.sqrt(((sig_x - sig_y) / 2)**2 + tau**2)  # radius of Mohr's circle
    sigma1 = avg + R      # max principal (≈ 0, nearly horizontal)
    sigma2 = avg - R      # min principal (≈ -1, vertical compression)

    # Principal angle: direction of sigma1
    theta = 0.5 * np.arctan2(2 * tau, sig_x - sig_y)
    vec1  = np.column_stack([np.cos(theta), np.sin(theta)])
    vec2  = np.column_stack([np.cos(theta + np.pi/2), np.sin(theta + np.pi/2)])

    return sigma1, sigma2, vec1, vec2


def stress_bending(nodes, Lx=2.0, Ly=1.0, P=1.0):
    """
    SIMPLY-SUPPORTED BEAM with a central point load.

    Physical meaning: a horizontal beam, resting on two supports at its ends,
    with a load P pushing down at the midpoint.

    Uses Euler-Bernoulli beam theory:
      - bending stress: sigma_x = M(x) * y_centroidal / I
      - shear stress:   tau    = V(x) * (h²/4 - y²) / (2I)  (parabolic)
    
    Then Mohr's circle gives the principal stresses and directions.
    """
    x, y = nodes[:, 0], nodes[:, 1]
    h  = Ly
    I  = h**3 / 12        # second moment of area (unit width)
    yc = y - h / 2        # distance from the neutral axis (midheight)

    # Bending moment and shear force (piecewise, symmetric about midspan)
    M = np.where(x <= Lx/2, (P/2)*x, (P/2)*(Lx - x))   # rises to peak at centre
    V = np.where(x <= Lx/2,  P/2,           -P/2)         # steps at centre

    # Stress components
    sig_x = M * yc / I                             # bending: tension below NA, compression above
    sig_y = np.zeros(len(nodes))                   # no vertical stress (thin beam assumption)
    tau   = (V / (2*I)) * ((h/2)**2 - yc**2)      # parabolic shear distribution

    # Mohr's circle → principal stresses
    avg    = (sig_x + sig_y) / 2
    R      = np.sqrt(((sig_x - sig_y) / 2)**2 + tau**2)
    sigma1 = avg + R
    sigma2 = avg - R

    theta = 0.5 * np.arctan2(2 * tau, sig_x - sig_y)
    vec1  = np.column_stack([np.cos(theta), np.sin(theta)])
    vec2  = np.column_stack([np.cos(theta + np.pi/2), np.sin(theta + np.pi/2)])

    return sigma1, sigma2, vec1, vec2


# --- Visualise stress fields ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='#080810')
stress_cases = [
    ('Radial',       generate_mesh(1.0,1.0,800), stress_radial,      dict(cx=0.5,cy=0.5)),
    ('Compression',  generate_mesh(1.0,2.0,900), stress_compression, dict(Lx=1.0,Ly=2.0)),
    ('Bending',      generate_mesh(2.0,1.0,900), stress_bending,     dict(Lx=2.0,Ly=1.0)),
]
for ax, (ttl, (n,t,v,a), sfn, skw) in zip(axes, stress_cases):
    s1,s2,v1,v2 = sfn(n, **skw)
    ax.set_facecolor('#080810')
    tc = ax.tricontourf(n[:,0],n[:,1],t,s1, levels=20, cmap='inferno', zorder=1)
    plt.colorbar(tc, ax=ax, label='σ₁').ax.yaxis.set_tick_params(color='white')
    step = max(1, len(n)//80)
    idx  = np.arange(0, len(n), step)
    scale = 0.04
    # Draw sigma1 direction arrows (white)
    ax.quiver(n[idx,0],n[idx,1],v1[idx,0]*scale,v1[idx,1]*scale,
              color='white',alpha=0.5,scale=1,scale_units='xy',width=0.003,zorder=3)
    ax.set_aspect('equal')
    ax.set_title(f'{ttl} — σ₁ field + directions', color='white', fontweight='bold')
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#3a3a5c')
plt.tight_layout(); plt.show()
print('Stress fields computed ✓')

---
## Step 3 — Stress-Guided Spanning Tree (Prim's Algorithm)

### Why Prim's algorithm?
Prim's is a classic graph algorithm for building a **Minimum Spanning Tree** — a tree that connects
every node with the minimum total edge cost. Here we customise the edge cost to follow stress directions.

### How it works (plain English)
1. Start at the seed node. Mark it visited.
2. Look at all mesh edges that connect a visited node to an unvisited node — these are "frontier" edges.
3. Pick the frontier edge with the **lowest weight** (best stress alignment, right direction, etc.).
4. Add that edge to the tree and mark the new node visited.
5. Repeat from step 2 until every node is visited.

Because we grow a *tree* (no cycles), there are **no overlaps**, and we are **guaranteed** to reach every node.

In [ ]:
def build_stress_spanning_tree(nodes, adj, vec1, vec2, sigma1, sigma2,
                                seed_node,
                                alpha=3.5,          # stress-direction alignment weight
                                beta=0.8,           # edge-length weight
                                gamma=0.8,          # stress-magnitude bonus weight
                                delta=2.5,          # backward-motion penalty weight
                                min_edge_frac=0.4): # min edge length (fraction of mean)
    """
    Build a stress-guided spanning tree using Prim's algorithm.

    Every node in the mesh is visited exactly once.
    The tree grows from `seed_node` outward, always choosing the cheapest
    available edge according to the custom weight formula.

    EDGE WEIGHT FORMULA:
    ─────────────────────────────────────────────────────────────────
    w = α · (1 - |align|)         → MISALIGNMENT COST
                                     align = |dot(edge_dir, σ₁_dir)|
                                     0 = edge ⊥ to σ₁ (bad)
                                     1 = edge ∥ to σ₁ (good)

      + β · edge_length           → LENGTH COST
                                     prefer shorter edges

      − γ · σ₁_magnitude         → STRESS MAGNITUDE BONUS
                                     subtract to reward high-stress paths

      + δ · backward_penalty      → BACKWARD PENALTY  ← KEY IMPROVEMENT
                                     penalise edges pointing opposite to
                                     the direction we just came from
    ─────────────────────────────────────────────────────────────────

    Parameters
    ----------
    alpha : float
        Stress-direction alignment weight.
        Higher → tree follows σ₁ direction more closely.
    beta : float
        Edge-length weight. Higher → shorter, denser branches.
    gamma : float
        Stress-magnitude bonus. Higher → tree preferentially hugs stress peaks.
    delta : float
        Backward-motion penalty.  ← NEW in v3
        Higher → branches never backtrack (removed ~20% of backward stubs).
        0 = no penalty (old behaviour), 2.5 = strong suppression.
    min_edge_frac : float
        Edges shorter than (min_edge_frac × mean_edge_length) are skipped.
        Prevents micro-branches from very closely spaced mesh nodes.

    Returns
    -------
    parent      : (N,) index of each node's parent in the tree  (-1 = seed)
    depth_arr   : (N,) depth from seed (0 = seed, 1 = seed's children, …)
    mst_edges   : list of (parent_idx, child_idx, depth) — every tree edge
    edge_sigma1 : (N,) σ₁ value recorded when each edge was added
    edge_sigma2 : (N,) σ₂ value recorded when each edge was added
    """

    # ── Estimate mean edge length (for minimum length threshold) ──
    sample_lens = []
    for u in list(adj.keys())[:80]:
        for v in adj[u]:
            sample_lens.append(np.linalg.norm(nodes[u] - nodes[v]))
    mean_len = np.mean(sample_lens)
    min_len  = min_edge_frac * mean_len   # edges shorter than this are skipped

    # ── Initialise Prim's algorithm data structures ─────────────
    N           = len(nodes)
    visited     = np.zeros(N, dtype=bool)    # True once a node is in the tree
    parent      = np.full(N, -1, dtype=int)  # parent node index (-1 = no parent)
    key         = np.full(N, np.inf)         # best edge weight seen so far
    depth_arr   = np.zeros(N, dtype=int)     # depth from seed
    edge_sigma1 = np.zeros(N)                # sigma1 at chosen edge
    edge_sigma2 = np.zeros(N)                # sigma2 at chosen edge

    # incoming_dir[i] = direction of the edge that arrived at node i
    # We use this to detect and penalise backward movement
    incoming_dir = np.zeros((N, 2))

    # Priority queue: (edge_weight, node_index)
    # heapq always pops the smallest element first
    key[seed_node] = 0.0
    heap      = [(0.0, seed_node)]
    mst_edges = []   # completed tree edges

    sig1_max = np.abs(sigma1).max() + 1e-12   # for normalisation

    # ── MAIN LOOP: grow the tree one edge at a time ─────────────
    while heap:

        # Pop the unvisited node with the cheapest edge from the frontier
        w_u, u = heapq.heappop(heap)

        # Skip if already visited (old heap entry that's now outdated)
        if visited[u]:
            continue

        # Mark this node as part of the tree
        visited[u] = True

        # Record the edge that brought us here
        if parent[u] >= 0:
            mst_edges.append((parent[u], u, depth_arr[u]))

        # ── Examine every mesh neighbour of node u ─────────────
        for v in adj[u]:

            if visited[v]:
                continue   # already in tree, ignore

            # Geometric properties of candidate edge u → v
            edge_vec = nodes[v] - nodes[u]
            edge_len = np.linalg.norm(edge_vec) + 1e-12
            edge_dir = edge_vec / edge_len    # unit direction

            # Skip edges that are too short → prevents micro-branches
            if edge_len < min_len:
                continue

            # ── COMPONENT 1: Stress-direction alignment ──────────
            # How parallel is this edge to the σ₁ direction at node u?
            # abs() because σ₁ has no preferred 'forward' end
            align_s1 = abs(np.dot(edge_dir, vec1[u]))   # 0=perp, 1=parallel
            align_s2 = abs(np.dot(edge_dir, vec2[u]))   # secondary direction

            # Blend: σ₁ dominates (80%), σ₂ adds branching texture (20%)
            align = 0.80 * align_s1 + 0.20 * align_s2

            # ── COMPONENT 2: Stress-magnitude bonus ─────────────
            # Average σ₁ at both ends of this edge
            avg_sig      = 0.5 * (abs(sigma1[u]) + abs(sigma1[v]))
            stress_bonus = gamma * avg_sig / sig1_max    # normalised to [0, γ]

            # ── COMPONENT 3: Backward-motion penalty ────────────
            # If this edge goes in the OPPOSITE direction to the one we
            # arrived from, penalise it. This stops branches from doubling
            # back on themselves (the 'stubs' in v2).
            inc      = incoming_dir[u]          # direction we came from
            inc_norm = np.linalg.norm(inc)

            if inc_norm > 1e-10:
                # forward_dot: +1 = same direction, -1 = opposite direction
                forward_dot      = np.dot(edge_dir, inc / inc_norm)
                # Only penalise negative (backward) component:
                backward_penalty = delta * max(0.0, -forward_dot)
            else:
                # Seed node has no incoming direction → no penalty
                backward_penalty = 0.0

            # ── TOTAL EDGE WEIGHT ────────────────────────────────
            w = (alpha * (1.0 - align)    # penalise misalignment
               + beta  * edge_len         # penalise length
               - stress_bonus             # reward high-stress zone
               + backward_penalty)        # penalise backward motion

            # ── Update the priority queue if this is a better edge ─
            if w < key[v]:
                key[v]          = w
                parent[v]       = u
                depth_arr[v]    = depth_arr[u] + 1
                incoming_dir[v] = edge_dir          # remember for next step
                edge_sigma1[v]  = abs(sigma1[u])    # store σ₁ for linewidth
                edge_sigma2[v]  = abs(sigma2[u])
                heapq.heappush(heap, (w, v))

    return parent, depth_arr, mst_edges, edge_sigma1, edge_sigma2


def compute_tree_depth(mst_edges, N, seed):
    """
    BFS from seed to assign proper branching depth to every node.
    Depth 0 = seed, Depth 1 = direct children of seed, etc.
    Also returns children dict: children[u] = [list of child node indices]
    """
    children = defaultdict(list)
    for p, c, _ in mst_edges:
        children[p].append(c)

    depth = np.full(N, -1, dtype=int)
    depth[seed] = 0
    queue = [seed]
    while queue:
        nxt = []
        for u in queue:
            for v in children[u]:
                if depth[v] < 0:
                    depth[v] = depth[u] + 1
                    nxt.append(v)
        queue = nxt

    return depth, children

print('Spanning tree functions defined ✓')

---
## Step 4 — Branch Thickness from σ₁ Magnitude

In a real branching structure (like a tree or blood vessel network),
**thicker branches carry more load**. We replicate this by mapping
the σ₁ magnitude at each edge to a linewidth:

```
high σ₁  →  thick branch  (main trunk, close to seed, high-stress zone)
low  σ₁  →  thin  branch  (leaf tip, far from seed, low-stress zone)
```

In [ ]:
def stress_to_linewidth(edge_sigma1, sigma1_all, lw_min=0.25, lw_max=4.5):
    """
    Convert sigma1 magnitude at each tree edge into a linewidth (in points).

    We normalise the sigma1 values to [0, 1] using the global min/max,
    then linearly map to [lw_min, lw_max].

    Parameters
    ----------
    edge_sigma1 : (N,) sigma1 recorded when each edge was added to the tree
    sigma1_all  : (N,) all nodal sigma1 values (for consistent normalisation)
    lw_min      : thinnest linewidth (leaf tips)
    lw_max      : thickest linewidth (near seed / high stress)

    Returns
    -------
    linewidths : (N,) array of linewidths, one per node
    """
    sig_min = np.abs(sigma1_all).min()
    sig_max = np.abs(sigma1_all).max() + 1e-12

    # Normalise to [0, 1]
    t = (edge_sigma1 - sig_min) / (sig_max - sig_min)
    t = np.clip(t, 0, 1)

    # Map to linewidth range
    return lw_min + (lw_max - lw_min) * t

print('Linewidth mapping function defined ✓')

---
## Step 5 — Visualisation

In [ ]:
def plot_full(nodes, tris, sigma1, sigma2, vec1, vec2,
              mst_edges, depth_arr, edge_sigma1,
              seed_node, title, Lx, Ly,
              bg='#080810', mesh_col='#252545',
              tree_cmap='turbo', node_cmap='plasma',
              lw_min=0.25, lw_max=4.5,
              show_stress_arrows=True,
              ax=None, fig=None):
    """
    Full visualisation of the branching pattern on the triangular mesh.

    What you'll see:
    ──────────────────────────────────────────────────────────
    ▪ Triangular mesh    — faint background grid
    ▪ Mesh nodes         — coloured by σ₁ magnitude (plasma colormap)
    ▪ Tree branches      — colour = branching depth from seed
                           thickness = σ₁ magnitude at that edge
    ▪ Stress arrows      — faint white arrows showing σ₁ directions
    ▪ Seed node          — white dot with gold ring
    ──────────────────────────────────────────────────────────
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, int(9*Ly/Lx)+1), facecolor=bg)
    ax.set_facecolor(bg)

    # ── 1. Mesh (faint grid lines) ────────────────────────────
    ax.triplot(nodes[:, 0], nodes[:, 1], tris,
               color=mesh_col, linewidth=0.30, alpha=0.75, zorder=1)

    # ── 2. Nodes coloured by σ₁ magnitude ────────────────────
    sig_norm = (sigma1 - sigma1.min()) / (sigma1.max() - sigma1.min() + 1e-12)
    sc = ax.scatter(nodes[:, 0], nodes[:, 1],
                    c=sig_norm, cmap=node_cmap, s=7,
                    vmin=0, vmax=1, zorder=2, alpha=0.80)

    # ── 3. Tree branches (colour=depth, thickness=σ₁) ─────────
    max_d   = max(depth_arr.max(), 1)
    cmap_tr = plt.cm.get_cmap(tree_cmap)
    linewidths = stress_to_linewidth(edge_sigma1, sigma1, lw_min, lw_max)

    # Group edges by depth for fast batch drawing with LineCollection
    segs_by_d = defaultdict(list)
    lws_by_d  = defaultdict(list)
    for p, c, _ in mst_edges:
        d = int(depth_arr[c])
        segs_by_d[d].append([nodes[p], nodes[c]])
        lws_by_d[d].append(linewidths[c])

    for d, segs in segs_by_d.items():
        color = cmap_tr(d / max_d)   # colour by depth
        lc = LineCollection(
            segs,
            colors=[color] * len(segs),
            linewidths=lws_by_d[d],   # width by stress magnitude
            zorder=5, alpha=0.93,
            capstyle='round', joinstyle='round'
        )
        ax.add_collection(lc)

    # ── 4. Stress direction arrows (faint) ───────────────────
    if show_stress_arrows:
        step  = max(1, len(nodes) // 90)
        idx   = np.arange(0, len(nodes), step)
        scale = min(Lx, Ly) * 0.032
        ax.quiver(
            nodes[idx, 0], nodes[idx, 1],
            vec1[idx, 0] * scale, vec1[idx, 1] * scale,
            color='white', alpha=0.13,
            scale=1, scale_units='xy', width=0.002, zorder=3
        )

    # ── 5. Seed node (gold ring) ─────────────────────────────
    ax.scatter([nodes[seed_node, 0]], [nodes[seed_node, 1]],
               c='white', s=140, zorder=11,
               edgecolors='#FFD700', linewidths=2.5)

    # ── 6. Colourbar for node σ₁ ─────────────────────────────
    cb = plt.colorbar(sc, ax=ax, fraction=0.028, pad=0.02)
    cb.set_label('σ₁ magnitude', color='white', fontsize=9)
    cb.ax.yaxis.set_tick_params(color='white')
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

    # ── 7. Branch depth legend ───────────────────────────────
    n_leg   = min(8, max_d + 1)
    handles = [
        mpatches.Patch(color=cmap_tr(d / max_d), label=f'Depth {d}')
        for d in np.linspace(0, max_d, n_leg, dtype=int)
    ]
    ax.legend(
        handles=handles, loc='upper right', fontsize=7,
        framealpha=0.3, labelcolor='white',
        facecolor='#1a1a2e', edgecolor='#3a3a5c',
        title='Branch depth', title_fontsize=7
    )

    # ── 8. Thickness annotation ──────────────────────────────
    ax.text(0.02, 0.02,
            f'thick = high σ₁ ({lw_max:.0f}pt)\nthin  = low  σ₁ ({lw_min:.1f}pt)',
            transform=ax.transAxes, color='#AAAACC',
            fontsize=8, va='bottom', alpha=0.85,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1a2e', alpha=0.5))

    # ── 9. Axes styling ──────────────────────────────────────
    ax.set_xlim(-0.02*Lx, 1.02*Lx)
    ax.set_ylim(-0.02*Ly, 1.02*Ly)
    ax.set_aspect('equal')
    ax.set_title(title, color='white', fontsize=12, fontweight='bold', pad=10)
    ax.tick_params(colors='white')
    ax.set_xlabel('x', color='white')
    ax.set_ylabel('y', color='white')
    for sp in ax.spines.values():
        sp.set_edgecolor('#3a3a5c')

    return fig, ax

print('Plotting function defined ✓')

---
## Step 6 — Run All Three Load Cases

In [ ]:
BG = '#080810'

# Define all cases: (name, Lx, Ly, n_nodes, stress_fn, kwargs, seed_pos, α, β, γ, δ)
cases = [
    ('radial',      1.0,1.0, 850, stress_radial,      dict(cx=0.5,cy=0.5), (0.5,0.5), 3.5,0.8,0.8,2.5),
    ('compression', 1.0,2.0, 950, stress_compression, dict(Lx=1.0,Ly=2.0), (0.5,1.0), 3.0,1.0,0.6,2.5),
    ('bending',     2.0,1.0, 950, stress_bending,     dict(Lx=2.0,Ly=1.0), (1.0,0.5), 3.0,0.9,0.7,2.5),
]

results = {}
for (name, Lx, Ly, n, sfn, skw, spos, alpha, beta, gamma, delta) in cases:
    print(f'\n── {name.upper()} ──')

    nodes, tris, valence, adj = generate_mesh(Lx, Ly, n_approx=n)
    print(f'  Mesh: {len(nodes)} nodes, {len(tris)} triangles')

    sig1, sig2, vec1, vec2 = sfn(nodes, **skw)

    # Find mesh node closest to the desired seed position
    seed = int(np.argmin(np.linalg.norm(nodes - np.array(spos), axis=1)))
    print(f'  Seed: node {seed} at {nodes[seed].round(3)}')

    parent, depth_arr, mst_edges, edge_sig1, edge_sig2 = build_stress_spanning_tree(
        nodes, adj, vec1, vec2, sig1, sig2,
        seed, alpha=alpha, beta=beta, gamma=gamma, delta=delta
    )

    print(f'  Tree edges: {len(mst_edges)} | Max depth: {depth_arr.max()}')
    print(f'  All nodes reached: {(depth_arr >= 0).all()}')

    results[name] = (nodes, tris, valence, sig1, sig2, vec1, vec2,
                     mst_edges, depth_arr, edge_sig1, edge_sig2, seed, Lx, Ly)

print('\nAll cases complete ✓')

---
## Step 7 — Plot Results

In [ ]:
# ── All three cases side by side ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 8), facecolor=BG)
fig.suptitle(
    'Principal Stress Spanning Tree  |  colour = branch depth  |  thickness = σ₁',
    color='white', fontsize=13, fontweight='bold', y=1.01
)
titles = {
    'radial':      'Radial / Biaxial\n(point-source stress)',
    'compression': 'Pure Compression\n(column  1×2)',
    'bending':     'Beam Bending\n(simply-supported  2×1)',
}
for ax, name in zip(axes, results):
    nodes,tris,_,sig1,sig2,vec1,vec2,mst,depth,es1,_,seed,Lx,Ly = results[name]
    plot_full(nodes,tris,sig1,sig2,vec1,vec2,mst,depth,es1,
              seed, titles[name], Lx, Ly, ax=ax, fig=fig)
plt.tight_layout(pad=1.5); plt.show()

In [ ]:
# ── Clean tree only (no mesh background) ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 8), facecolor=BG)
fig.suptitle('Clean Branching Tree  |  thickness ∝ σ₁ magnitude',
             color='white', fontsize=13, fontweight='bold', y=1.01)

cmaps = ['turbo', 'cool', 'plasma']
for ax, name, cmap_name in zip(axes, results, cmaps):
    nodes,tris,_,sig1,sig2,vec1,vec2,mst,depth,es1,_,seed,Lx,Ly = results[name]
    ax.set_facecolor(BG)
    # Very faint mesh
    ax.triplot(nodes[:,0],nodes[:,1],tris, color='#181828', lw=0.25, alpha=0.5, zorder=1)
    max_d = max(depth.max(),1); cmap2 = plt.cm.get_cmap(cmap_name)
    lws   = stress_to_linewidth(es1, sig1, 0.25, 5.0)
    segs_d = defaultdict(list); lws_d = defaultdict(list)
    for p,c,_ in mst:
        d=int(depth[c]); segs_d[d].append([nodes[p],nodes[c]]); lws_d[d].append(lws[c])
    for d,segs in segs_d.items():
        ax.add_collection(LineCollection(segs, colors=[cmap2(d/max_d)]*len(segs),
            linewidths=lws_d[d], zorder=5, alpha=0.95, capstyle='round', joinstyle='round'))
    ax.scatter([nodes[seed,0]],[nodes[seed,1]], c='white', s=160, zorder=11,
               edgecolors='#FFD700', linewidths=3.0)
    ax.set_xlim(-0.02*Lx,1.02*Lx); ax.set_ylim(-0.02*Ly,1.02*Ly)
    ax.set_aspect('equal')
    ax.set_title(titles[name], color='white', fontsize=12, fontweight='bold', pad=8)
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#3a3a5c')
plt.tight_layout(pad=1.5); plt.show()

---
## Step 8 — Delta (δ) Parameter Study
δ controls how strongly backward branches are suppressed.

In [ ]:
nodes_r,tris_r,_,sig1_r,sig2_r,vec1_r,vec2_r,_,_,_,_,seed_r,_,_ = results['radial']
_, _, _, adj_r = generate_mesh(1.0, 1.0, n_approx=850)

delta_vals = [0.0, 0.5, 1.0, 1.5, 2.5, 4.0]
fig, axes  = plt.subplots(2, 3, figsize=(18, 13), facecolor=BG)
fig.suptitle('Effect of Backward-Penalty δ  (α=3.5  β=0.8  γ=0.8)',
             color='white', fontsize=13, fontweight='bold', y=1.01)

for ax, delta_v in zip(axes.ravel(), delta_vals):
    _,depth_v,mst_v,es1_v,_ = build_stress_spanning_tree(
        nodes_r,adj_r,vec1_r,vec2_r,sig1_r,sig2_r,seed_r,
        alpha=3.5,beta=0.8,gamma=0.8,delta=delta_v)

    # Count backward edges for this delta
    parent_v = {c:p for p,c,_ in mst_v}
    bwd = sum(1 for p,c,_ in mst_v
              if parent_v.get(p,-1)>=0 and
              np.linalg.norm(nodes_r[p]-nodes_r[parent_v[p]])>1e-10 and
              np.linalg.norm(nodes_r[c]-nodes_r[p])>1e-10 and
              np.dot((nodes_r[c]-nodes_r[p])/np.linalg.norm(nodes_r[c]-nodes_r[p]),
                     (nodes_r[p]-nodes_r[parent_v[p]])/np.linalg.norm(nodes_r[p]-nodes_r[parent_v[p]]))<-0.3)

    ax.set_facecolor(BG)
    ax.triplot(nodes_r[:,0],nodes_r[:,1],tris_r,color='#252545',lw=0.28,alpha=0.6,zorder=1)
    sig_n=(sig1_r-sig1_r.min())/(sig1_r.max()-sig1_r.min()+1e-12)
    ax.scatter(nodes_r[:,0],nodes_r[:,1],c=sig_n,cmap='plasma',s=6,vmin=0,vmax=1,zorder=2,alpha=0.75)
    max_dv=max(depth_v.max(),1); lws_v=stress_to_linewidth(es1_v,sig1_r,0.25,4.0)
    cmap_v=plt.cm.get_cmap('turbo')
    ds_v=defaultdict(list); lw_v=defaultdict(list)
    for p2,c2,_ in mst_v:
        d2=int(depth_v[c2]); ds_v[d2].append([nodes_r[p2],nodes_r[c2]]); lw_v[d2].append(lws_v[c2])
    for d2,segs2 in ds_v.items():
        ax.add_collection(LineCollection(segs2,colors=[cmap_v(d2/max_dv)]*len(segs2),
            linewidths=lw_v[d2],zorder=5,alpha=0.93,capstyle='round'))
    ax.scatter([nodes_r[seed_r,0]],[nodes_r[seed_r,1]],c='white',s=100,zorder=11,
               edgecolors='#FFD700',linewidths=2)
    ax.set_xlim(-0.02,1.02); ax.set_ylim(-0.02,1.02); ax.set_aspect('equal')
    ax.set_title(f'δ = {delta_v}  →  backward stubs: {100*bwd/len(mst_v):.1f}%',
                 color='white', fontsize=10, fontweight='bold')
    ax.tick_params(colors='white', labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#3a3a5c')
plt.tight_layout(pad=1.2); plt.show()

---
## Summary — Parameter Tuning Guide

| Parameter | Effect | Typical range |
|-----------|--------|---------------|
| `alpha` α | How strongly branches follow σ₁ direction | 1.0 – 5.0 |
| `beta`  β | Preference for shorter edges (denser local branching) | 0.5 – 2.0 |
| `gamma` γ | Bonus for high-stress zones | 0.3 – 1.5 |
| `delta` δ | Suppression of backward-going stubs | 0.0 – 4.0 |
| `min_edge_frac` | Minimum edge length (removes micro-branches) | 0.2 – 0.6 |
| `seed_pos` | Where the tree starts growing from | any (x,y) in domain |
| `lw_min/max` | Linewidth range for stress-based branch thickness | 0.1 – 6.0 |

### Next steps
- Replace analytical stress with **real FEA results** (import from CSV/VTK)
- Grow **two overlapping trees** (one for σ₁, one for σ₂) — orthogonal network
- Use a **circular or arbitrary domain** by masking nodes outside a boundary
- Export branch geometry as DXF/SVG for fabrication